# z003 - Lags y delta lags (orden 1 y 2)

**Objetivo:** ver si agregar la historia del cliente mejora la ganancia en validación.

- **lag1 / lag2:** valor de la variable 1 y 2 meses antes.
- **delta1 / delta2:** cuánto cambió respecto de 1 y 2 meses antes (actual - lag).

**Corridas** (mismo esquema que z002: train 202104, valid 202106, misma semilla):

| Corrida | Variables |
|---|---|
| A. Base | originales |
| B. + orden 1 | originales + lag1 + delta1 |
| C. + orden 1 y 2 | originales + lag1, lag2, delta1, delta2 |

Ojo: los datos arrancan en 202103, así que en el train (202104) **lag2 queda todo vacío (NaN)**.

In [1]:
# --- 1. Librerías y parámetros ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
UMBRAL = COSTO_ESTIMULO / (GANANCIA_ACIERTO + COSTO_ESTIMULO)   # = 0.025

MESES_TRAIN = [202104]
MES_VALID   = 202106
SEMILLA     = 501419

params = {
    "objective": "binary",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "seed": SEMILLA,
    "verbose": -1,
}

In [2]:
# --- 2. Cargar datos y armar el target binario ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

# Variables originales (igual que z002: numero_de_cliente se usa)
no_usar = ["foto_mes", "clase_ternaria", "clase_binaria"]
originales = [c for c in df.columns if c not in no_usar]

# Columnas a las que les calculamos historia (todas menos el ID del cliente)
cols = [c for c in originales if c != "numero_de_cliente"]
print("Variables originales:", len(originales), "| con lags:", len(cols))

Variables originales: 153 | con lags: 152


In [3]:
# --- 3. Calcular lags y deltas ---
# Ordenamos por cliente y mes: así "el anterior" de cada fila es el mes previo del mismo cliente
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]

nuevas = []
for k in [1, 2]:
    # shift(k) = valor de k meses antes dentro de cada cliente (NaN si no hay)
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")      # cuánto cambió
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]

df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo   # liberamos memoria

# Listas de variables para cada corrida
lag1   = [f"{c}_lag1" for c in cols] + [f"{c}_delta1" for c in cols]
lag2   = [f"{c}_lag2" for c in cols] + [f"{c}_delta2" for c in cols]
corridas = {
    "A. Base":          originales,
    "B. + orden 1":     originales + lag1,
    "C. + orden 1 y 2": originales + lag1 + lag2,
}
print("Columnas totales:", df.shape[1])
for nombre, v in corridas.items():
    print(nombre, "->", len(v), "variables")

Columnas totales: 764
A. Base -> 153 variables
B. + orden 1 -> 457 variables
C. + orden 1 y 2 -> 761 variables


In [4]:
# --- 4. Función: entrenar, predecir validación y calcular métricas ---
def evaluar(variables, semilla=SEMILLA, meses_train=MESES_TRAIN, mes_valid=MES_VALID):
    train = df[df["foto_mes"].isin(meses_train)]
    valid = df[df["foto_mes"] == mes_valid]
    y_valid = valid["clase_binaria"].values

    p = {**params, "seed": semilla}   # mismos parámetros, cambiando solo la semilla
    datos = lgb.Dataset(train[variables], label=train["clase_binaria"])
    modelo = lgb.train(p, datos, num_boost_round=300)
    prob = modelo.predict(valid[variables])

    # Ganancia acumulada ordenando de mayor a menor probabilidad
    orden = np.argsort(prob)[::-1]
    ganancia_cliente = np.where(y_valid == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    ganancia_acum = np.cumsum(ganancia_cliente[orden])
    mejor_n = int(np.argmax(ganancia_acum)) + 1

    # AUC y KS a mano (como en z002)
    y_ord = y_valid[orden]
    tpr = np.cumsum(y_ord) / y_ord.sum()
    fpr = np.cumsum(1 - y_ord) / (len(y_ord) - y_ord.sum())

    estimulados = prob >= UMBRAL
    r = {
        "n_corte_0025":    int(estimulados.sum()),
        "ganancia_0025_M": round(ganancia_cliente[estimulados].sum() / 1e6, 1),
        "mejor_n":         mejor_n,
        "ganancia_max_M":  round(ganancia_acum[mejor_n - 1] / 1e6, 1),
        "ganancia_top10000_M": round(ganancia_acum[10_000 - 1] / 1e6, 1),
        "ganancia_top12000_M": round(ganancia_acum[12_000 - 1] / 1e6, 1),
        "prob_en_corte":   round(float(prob[orden][mejor_n - 1]), 4),
        "auc":             round(float(np.trapezoid(np.r_[0, tpr], np.r_[0, fpr])), 3),
        "ks":              round(float((tpr - fpr).max()), 3),
    }
    return r, modelo, ganancia_acum

In [5]:
# --- 5. Correr las 3 corridas ---
resultados, modelos = {}, {}
for nombre, variables in corridas.items():
    resultados[nombre], modelos[nombre], _ = evaluar(variables)
    print(nombre, "listo")

tabla = pd.DataFrame(resultados).T
tabla

A. Base listo


B. + orden 1 listo


C. + orden 1 y 2 listo


,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,ganancia_top10000_M,ganancia_top12000_M,prob_en_corte,auc,ks
A. Base,6610.0,297.8,9299.0,330.6,321.2,323.4,0.0166,0.881,0.620
B. + orden 1,5749.0,278.6,12464.0,350.2,336.6,339.9,0.0091,0.884,0.628
C. + orden 1 y 2,5749.0,278.6,12464.0,350.2,336.6,339.9,0.0091,0.884,0.628


In [6]:
# --- 6. Importancia de variables en la corrida C ---
modelo_c = modelos["C. + orden 1 y 2"]
importancia = pd.DataFrame({
    "variable": corridas["C. + orden 1 y 2"],
    "importancia": modelo_c.feature_importance(importance_type="gain"),
}).sort_values("importancia", ascending=False).reset_index(drop=True)
importancia["pct"] = (importancia["importancia"] / importancia["importancia"].sum() * 100).round(2)

# ¿Qué parte de la importancia se llevan los lags y deltas?
def tipo(v):
    for t in ["lag1", "lag2", "delta1", "delta2"]:
        if v.endswith("_" + t):
            return t
    return "original"
importancia["tipo"] = importancia["variable"].map(tipo)
print(importancia.groupby("tipo")["pct"].sum().sort_values(ascending=False))

importancia.head(20)

tipo
original    50.38
lag1        26.44
delta1      23.24
delta2       0.00
lag2         0.00
Name: pct, dtype: float64


,variable,importancia,pct,tipo
0,ctrx_quarter,16317.418530,15.55,original
1,ctrx_quarter_lag1,4874.588043,4.65,lag1
2,mcuentas_saldo,3087.389995,2.94,original
3,mprestamos_personales,1944.200782,1.85,original
4,mactivos_margen,1800.361119,1.72,original
5,mrentabilidad_annual_lag1,1708.245391,1.63,lag1
6,ctarjeta_master,1674.969971,1.60,original
7,cprestamos_personales,1664.528407,1.59,original
8,mpasivos_margen,1661.477535,1.58,original
9,mpayroll,1446.791571,1.38,original


In [7]:
# --- 7. Anotar las corridas en resultados_experimentos.csv ---
ruta_res = "resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
# Si se vuelve a correr, borramos las filas viejas de z003 para no duplicarlas
res = res[~res["descripcion"].str.startswith("z003")]

for nombre, r in resultados.items():
    nueva = {
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z003 lags {nombre}, solo BAJA+2",
        "meses_train": str(MESES_TRAIN),
        "mes_valid": MES_VALID,
        "semilla": SEMILLA,
        "params": str(params),
        **r,
    }
    res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)

res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(3)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
3,2026-10-07 18:25,"z003 lags A. Base, solo BAJA+2",[202104],202106,501419,"{'objective': 'binary', 'learning_rate': 0.05,...",6610.0,297.8,9299.0,330.6,0.0166,0.881,0.620,NaN,NaN,<NA>,NaN,321.2,323.4
4,2026-10-07 18:25,"z003 lags B. + orden 1, solo BAJA+2",[202104],202106,501419,"{'objective': 'binary', 'learning_rate': 0.05,...",5749.0,278.6,12464.0,350.2,0.0091,0.884,0.628,NaN,NaN,<NA>,NaN,336.6,339.9
5,2026-10-07 18:25,"z003 lags C. + orden 1 y 2, solo BAJA+2",[202104],202106,501419,"{'objective': 'binary', 'learning_rate': 0.05,...",5749.0,278.6,12464.0,350.2,0.0091,0.884,0.628,NaN,NaN,<NA>,NaN,336.6,339.9


## 5 semillas: ¿la mejora de los lags es real o es ruido?

Una sola semilla puede engañar (A dio 330,6 M acá y 335,8 M en z002 con el mismo modelo).
Repetimos con 5 semillas y promediamos.

1. **A vs B** con el esquema de siempre (train 202104, valid 202106).
2. **B vs C** con train 202105 (ese mes sí tiene lag2), valid 202106: la única forma de medir el orden 2.

In [8]:
# --- 8. Correr cada comparación con las 5 semillas ---
SEMILLAS = [501419, 623353, 786349, 834569, 946331]

comparaciones = {
    # nombre: (variables, meses de train)
    "A. Base (train 202104)":          (corridas["A. Base"],          [202104]),
    "B. + orden 1 (train 202104)":     (corridas["B. + orden 1"],     [202104]),
    "B. + orden 1 (train 202105)":     (corridas["B. + orden 1"],     [202105]),
    "C. + orden 1 y 2 (train 202105)": (corridas["C. + orden 1 y 2"], [202105]),
}

filas, curvas = [], {}
for nombre, (variables, meses) in comparaciones.items():
    curvas[nombre] = []
    for semilla in SEMILLAS:
        r, _, ganancia_acum = evaluar(variables, semilla=semilla, meses_train=meses)
        filas.append({"corrida": nombre, "semilla": semilla, **r})
        curvas[nombre].append(ganancia_acum)
    print(nombre, "listo")

por_semilla = pd.DataFrame(filas)
por_semilla.pivot(index="semilla", columns="corrida", values="ganancia_max_M")

A. Base (train 202104) listo


B. + orden 1 (train 202104) listo


B. + orden 1 (train 202105) listo


C. + orden 1 y 2 (train 202105) listo


corrida,A. Base (train 202104),B. + orden 1 (train 202104),B. + orden 1 (train 202105),C. + orden 1 y 2 (train 202105)
semilla,,,,
501419,330.6,350.2,374.7,366.8
623353,344.1,351.8,356.1,364.2
786349,331.7,350.8,357.2,368.6
834569,336.2,340.7,363.4,354.3
946331,340.6,342.3,360.2,362.7


In [9]:
# --- 9. Promedios y curva de ganancia promedio ---
# Promedio y desvío de cada métrica entre las 5 semillas
resumen = (por_semilla.groupby("corrida", sort=False)
           [["ganancia_max_M", "ganancia_top10000_M", "ganancia_top12000_M", "mejor_n", "auc"]]
           .agg(["mean", "std"]).round(1))

# Curva promedio (idea de la tutoría): promediamos las 5 curvas y buscamos su máximo
for nombre, lista in curvas.items():
    curva_prom = np.mean(lista, axis=0)
    n = int(np.argmax(curva_prom)) + 1
    resumen.loc[nombre, ("curva_prom", "mejor_n")] = n
    resumen.loc[nombre, ("curva_prom", "ganancia_M")] = round(curva_prom[n - 1] / 1e6, 1)

# ¿En cuántas semillas gana cada versión?
g = por_semilla.pivot(index="semilla", columns="corrida", values="ganancia_max_M")
print("B le gana a A en", (g["B. + orden 1 (train 202104)"] > g["A. Base (train 202104)"]).sum(), "de 5 semillas")
print("C le gana a B en", (g["C. + orden 1 y 2 (train 202105)"] > g["B. + orden 1 (train 202105)"]).sum(), "de 5 semillas")
resumen

B le gana a A en 5 de 5 semillas
C le gana a B en 3 de 5 semillas


ganancia_max_M      ganancia_top10000_M       \
                                          mean  std                mean  std   
corrida                                                                        
A. Base (train 202104)                   336.6  5.8               326.0  7.7   
B. + orden 1 (train 202104)              347.2  5.2               334.4  5.7   
B. + orden 1 (train 202105)              362.3  7.5               348.0  8.5   
C. + orden 1 y 2 (train 202105)          363.3  5.5               344.7  4.6   

                                ganancia_top12000_M       mejor_n          \
                                               mean  std     mean     std   
corrida                                                                     
A. Base (train 202104)                        330.4  8.0  11342.4  1992.1   
B. + orden 1 (train 202104)                   340.6  4.2  11814.8   983.2   
B. + orden 1 (train 202105)                   356.0  8.0  12792.6  1897.0   
C. + orden 1 y 2 (train 202105)               355.5  4.3  13924.8  1702.0   

                                 auc      curva_prom             
                                mean  std    mejor_n ganancia_M  
corrida                                                          
A. Base (train 202104)           0.9  0.0    12064.0      331.5  
B. + orden 1 (train 202104)      0.9  0.0    11420.0      343.3  
B. + orden 1 (train 202105)      0.9  0.0    11687.0      359.3  
C. + orden 1 y 2 (train 202105)  0.9  0.0    11836.0      357.6

In [10]:
# --- 10. Anotar los promedios en resultados_experimentos.csv ---
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
promedios = por_semilla.groupby("corrida", sort=False).mean(numeric_only=True).drop(columns="semilla")

for nombre, (variables, meses) in comparaciones.items():
    r = promedios.loc[nombre].round(4).to_dict()
    nueva = {
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z003 5 semillas (promedio) {nombre}",
        "meses_train": str(meses),
        "mes_valid": MES_VALID,
        "semilla": str(SEMILLAS),
        "params": str(params),
        **r,
    }
    res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)

res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(4)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
6,2026-10-07 18:29,z003 5 semillas (promedio) A. Base (train 202104),[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.05,...",6669.4,299.04,11342.4,336.64,0.0133,0.8810,0.6188,NaN,NaN,<NA>,NaN,326.04,330.44
7,2026-10-07 18:29,z003 5 semillas (promedio) B. + orden 1 (train...,[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.05,...",5742.2,277.46,11814.8,347.16,0.0100,0.8844,0.6304,NaN,NaN,<NA>,NaN,334.40,340.56
8,2026-10-07 18:29,z003 5 semillas (promedio) B. + orden 1 (train...,[202105],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.05,...",4886.2,265.36,12792.6,362.32,0.0081,0.8888,0.6438,NaN,NaN,<NA>,NaN,348.04,355.96
9,2026-10-07 18:29,z003 5 semillas (promedio) C. + orden 1 y 2 (t...,[202105],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.05,...",4306.0,233.36,13924.8,363.32,0.0065,0.8886,0.6436,NaN,NaN,<NA>,NaN,344.74,355.52


## Conclusiones (completar después de correr)

- ¿B o C le ganan a A? ¿Por cuánto?
- ¿Qué parte de la importancia se llevan los lags/deltas?
- Recordar: en train (202104) lag2 está vacío → en el modelo final (train hasta 202106) puede rendir más.